# Programmieraufgabe 1: (gedämpftes) Newton-Verfahren

**Abgabe in den Programmiertutorien am 14. und 15. Mai 2025.**

Benötigte Module für dieses Notebook:

In [1]:
import numpy as np

In diesem Notebook wollen wir das Newton-Verfahren und das gedämpfte Newton-Verfahren zur Approximation einer Nullstelle der Funktion
$$f: \mathbb{R}^2 \to \mathbb{R}^2, \quad x = (x_1,x_2) \mapsto \begin{pmatrix} 1 - \frac{2}{\exp(x_1-x_2)+1} \\ x_2^3 + x_2 -2 \end{pmatrix}$$
testen.

**(a) Schreiben Sie jeweils eine Prozedur, die für einen Vektor $x\in\mathbb{R}^2$ die Funktion bzw. deren Ableitung an der Stelle $x$ berechnet und als `numpy`-array zurückgibt.**

In [2]:
def f(x):
    f1 = 1 - 2/( np.exp(x[0]-x[1]) + 1)
    f2 = x[1]**3 + x[1] - 2
    return np.array([f1,f2])

In [3]:
def fprime(x):
    temp = np.exp(x[0]-x[1])
    d1_f1 = 2/(temp+1)**2 * temp
    d2_f1 = -d1_f1
    d1_f2 = 0
    d2_f2 = 3*x[1]**2 + 1
    return np.array([[d1_f1,d2_f1],[d1_f2,d2_f2]])

In [4]:
x = np.array([0.0,0.0])
print(f(x))
print(fprime(x))

[ 0. -2.]
[[ 0.5 -0.5]
 [ 0.   1. ]]


**(b) Schreiben Sie eine Prozedur, die das Newton-Verfahren mit den in der Vorlesung besprochenen Kovergenz- und Abbruchkriterien auf eine Funktion $f:\mathbb{R}^n \to \mathbb{R}^n$ für beliebiges $n\in\mathbb{N}$ anwendet.**

Ihrer Prozedur soll folgende Eingabedaten haben:
- Einen Vektor `x0`, der den Startwert $x^{(0)}\in\mathbb{R}^n$ für das Newton-Verfahren enthält.
- Zwei Prozeduren `f` und `fprime`, mit denen Funktions- und Ableitungswerte von $f$ berechnet werden können.
- Eine Toleranz `tol` und eine maximale Zahl an Iterationen `kMax`, mit denen die Konvergenz- und Abbruchkriterien gesteuert werden können.

Berücksichtigen Sie außerdem folgendes:
- Die im Newton-Verfahren auftretenden Gleichungssysteme können Sie mit dem in `numpy` enthaltenen LGS-Löser `np.linalg.solve` lösen.
- Am Ende der Prozedur sollen alle berechneten Iterierten $x^{(k)}$, $k=0,1,2,...$ an das Hauptprogramm zurückgegeben werden. Geben Sie außerdem eine Meldung aus, ob das Verfahren erfolgreich konvergiert ist, oder ob es wegen Divergenz bzw. wegen Erreichen der maximalen Iterationszahl abgebrochen hat.

_Hinweis:_ Sie können alle Iterierten zum Beispiel als Zeilen in einer gemeinsamen Matrix speichern. Jedes Mal, wenn eine neue Iterierte berechnet wurde, wird die Matrix dementsprechend um eine Zeile ergänzt. Dazu ist der Befehl `np.vstack` hilfreich. Beispielsweise werden drei Matrizen `A1,A2,A3` derselben Breite mit dem Aufruf `np.vstack((A1,A2,A3))` übereinander gestapelt.

In [5]:
# Beachte: x0 muss als Zeilenvektor übergeben werden
def newton(x0,f,fprime,tol,kMax):
    delta_x = 2*tol
    k = 0
    x = np.array([x0]) # Matrix, die alle Iterierten als Zeilen enthält
    while np.linalg.norm(delta_x)>tol:
        # neue Newton-Iterierte berechnen und speichern
        delta_x = np.linalg.solve(fprime(x[k,:]),-f(x[k,:]))
        x_new = x[k,:] + delta_x
        x = np.vstack((x,x_new))
        
        # Abbruchkriterium prüfen
        delta_x_dash = np.linalg.solve(fprime(x[k,:]),-f(x_new))
        if np.linalg.norm(delta_x_dash) > np.linalg.norm(delta_x):
            print('Iteration scheint zu divergieren --> Abbruch!')
            return x
            
        # k erhöhen
        k += 1
        if  k == kMax:
            print('Maximale Anzahl an Iterationen erreicht --> Abbruch!')
            return x
                
    print('Newton-Verfahren konvergiert!')
    return x
    

**(c) Testen Sie Ihre Prozedur für die obige Funktion $f$ mit den Startvektoren $x^{(0)} = (4,2)^T$ und $x^{(0)} = (4,-4)^T$. Verwenden Sie die Parameter `tol = 1e-8` und `kMax = 20`.**

Geben Sie alle Iterierten aus. Für welchen Startvektor konvergiert das Verfahren? Vergewissern Sie sich im Falle der Konvergenz anhand der Funktionsvorschrift von $f$, dass Sie tatsächlich eine Nullstelle von $f$ erhalten haben. Geben Sie in dem Fall auch den Fehler der einzelnen Iterierten aus. Können Sie die erwartete, quadratische Konvergenz erkennen?

Startwert $x^{(0)} = (4,2)^T$:

In [6]:
x0 = np.array([4,2])
tol = 1e-8
kMax = 20
x = newton(x0,f,fprime,tol,kMax)

x_ex = np.array([1,1])
print('k | Iterierte x^(k)           | Fehler')
for k in range(x.shape[0]):
    print(f"{k} | {x[k,0]:+.8f} , {x[k,1]:+.8f} | {np.linalg.norm(x[k,:]-x_ex):.2e}")

Newton-Verfahren konvergiert!
k | Iterierte x^(k)           | Fehler
0 | +4.00000000 , +2.00000000 | 3.16e+00
1 | -0.24224502 , +1.38461538 | 1.30e+00
2 | +1.90139076 , +1.08258613 | 9.05e-01
3 | +0.91017052 , +1.00478035 | 9.00e-02
4 | +1.00015828 , +1.00001707 | 1.59e-04
5 | +1.00000000 , +1.00000000 | 3.09e-10
6 | +1.00000000 , +1.00000000 | 0.00e+00


Startwert $x^{(0)} = (4,-4)^T$:

In [7]:
x0 = np.array([4,-4])
tol = 1e-8
kMax = 20
x = newton(x0,f,fprime,tol,kMax)

x_ex = np.array([1,1])
print('k | Iterierte x^(k)')
for k in range(x.shape[0]):
    print(f"{k} | {x[k,0]:+.8f} , {x[k,1]:+.8f}")

Iteration scheint zu divergieren --> Abbruch!
k | Iterierte x^(k)
0 | +4.00000000 , -4.00000000
1 | -1485.05025436 , -2.57142857


**(d) Kopieren Sie das Newton-Verfahren von oben und ändern Sie es zum gedämpften Newton-Verfahren ab. Der Parameter $\lambda_{\min}$ soll dabei als zusätzlicher Eingabeparameter übergeben werden, während im ersten Schritt immer mit dem Wert $\lambda = 1$ gestartet wird. Geben Sie zusätzlich zu den Iterierten auch die verwendeten Dämpungsparameter an das Hauptprogramm zurück.**

_Hinweis:_ Das Schlüsselwort `lambda` hat in Python eine ganz besondere Bedeutung und sollte daher nicht als Variablenname genutzt werden.

In [9]:
# Beachte: x0 muss als Zeilenvektor übergeben werden
def newton_damped(x0,f,fprime,tol,kMax,lambdaMin):
    delta_x = 2*tol
    k = 0
    lam = 1.0 # Dämpfungsparameter
    lambdaAll = 0
    x = np.array([x0]) # Matrix, die alle Iterierten als Zeilen enthält
    while np.linalg.norm(delta_x)>tol:
        # Ersten Vorschlag für neue Newton-Iterierte berechnen
        delta_x = np.linalg.solve(fprime(x[k,:]),-f(x[k,:]))
        x_new = x[k,:] + lam * delta_x

        # Überprüfe Resultat, verkleinere ggf. den Dämpfungsparameter
        delta_x_dash = np.linalg.solve(fprime(x[k,:]),-f(x_new))
        while np.linalg.norm(delta_x_dash) > np.linalg.norm(delta_x):
            lam = lam/2
            if lam < lambdaMin:
                print('Dämpungsparameter muss zu klein gewählt werden --> Abbruch!')
                return x
            x_new = x[k,:] + lam * delta_x
            delta_x_dash = np.linalg.solve(fprime(x[k,:]),-f(x_new))
        lambdaAll = np.vstack((lambdaAll,lam))

        # Speichere neue Iterierte
        x = np.vstack((x,x_new))
        # Erhöhe lambda
        lam = min(1,2*lam)
        
        # k erhöhen
        k += 1
        if  k == kMax:
            print('Maximale Anzahl an Iterationen erreicht --> Abbruch!')
            return x
                
    print('Gedämpftes Newton-Verfahren konvergiert!')
    return x,lambdaAll
    

**(e) Wiederholen Sie Teil (c) mit dem gedämpften Newton-Verfahren. Verwenden Sie dabei $\lambda_{\min}=10^{-10}$ und geben Sie die Werte des Dämpungsparameters aus.**

Startwert $x^{(0)} = (4,2)^T$:

In [10]:
x0 = np.array([4,2])
tol = 1e-8
kMax = 20
lambdaMin = 1e-10
x,lambdaAll = newton_damped(x0,f,fprime,tol,kMax,lambdaMin)

x_ex = np.array([1,1])
print(' k | Iterierte x^(k)           | lambda   | Fehler  ')
for k in range(x.shape[0]):
    print(f"{k:2.0f} | {x[k,0]:+.8f} , {x[k,1]:+.8f} | {lambdaAll[k,0]:+.5f} | {np.linalg.norm(x[k,:]-x_ex):.2e}")

Gedämpftes Newton-Verfahren konvergiert!
 k | Iterierte x^(k)           | lambda   | Fehler  
 0 | +4.00000000 , +2.00000000 | +0.00000 | 3.16e+00
 1 | -0.24224502 , +1.38461538 | +1.00000 | 1.30e+00
 2 | +1.90139076 , +1.08258613 | +1.00000 | 9.05e-01
 3 | +0.91017052 , +1.00478035 | +1.00000 | 9.00e-02
 4 | +1.00015828 , +1.00001707 | +1.00000 | 1.59e-04
 5 | +1.00000000 , +1.00000000 | +1.00000 | 3.09e-10
 6 | +1.00000000 , +1.00000000 | +1.00000 | 0.00e+00


Startwert $x^{(0)} = (4,-4)^T$:

In [11]:
x0 = np.array([4,-4])
tol = 1e-8
kMax = 20
lambdaMin = 1e-10
x,lambdaAll = newton_damped(x0,f,fprime,tol,kMax,lambdaMin)

x_ex = np.array([1,1])
print(' k | Iterierte x^(k)           | lambda   | Fehler  ')
for k in range(x.shape[0]):
    print(f"{k:2.0f} | {x[k,0]:+.8f} , {x[k,1]:+.8f} | {lambdaAll[k,0]:+.5f} | {np.linalg.norm(x[k,:]-x_ex):.2e}")

Gedämpftes Newton-Verfahren konvergiert!
 k | Iterierte x^(k)           | lambda   | Fehler  
 0 | +4.00000000 , -4.00000000 | +0.00000 | 5.83e+00
 1 | -7.63320511 , -3.98883929 | +0.00781 | 9.97e+00
 2 | -7.31224568 , -3.96657020 | +0.01562 | 9.68e+00
 3 | -6.82503331 , -3.92224069 | +0.03125 | 9.24e+00
 4 | -6.16938737 , -3.83441038 | +0.06250 | 8.65e+00
 5 | -5.35746869 , -3.66201729 | +0.12500 | 7.88e+00
 6 | -4.36717242 , -3.32991961 | +0.25000 | 6.90e+00
 7 | -3.13385084 , -2.71335509 | +0.50000 | 5.56e+00
 8 | -1.63142036 , -1.64392215 | +1.00000 | 3.73e+00
 9 | -0.75601228 , -0.75601195 | +1.00000 | 2.48e+00
10 | +0.41839333 , +0.41839333 | +1.00000 | 8.23e-01
11 | +0.91288792 , +0.91288792 | +0.50000 | 1.23e-01
12 | +1.00612654 , +1.00612654 | +1.00000 | 8.66e-03
13 | +1.00002801 , +1.00002801 | +1.00000 | 3.96e-05
14 | +1.00000000 , +1.00000000 | +1.00000 | 8.32e-10
15 | +1.00000000 , +1.00000000 | +1.00000 | 0.00e+00
